# Khmer Legal Semantic Search

Run Phase 5 first. This notebook embeds a Khmer query with the selected model and searches the matching ChromaDB collection.

In [ ]:
import gc
import json
import logging
import sys
import unicodedata
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
CHROMA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data' / 'processed' / 'chroma_db'
logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel(logging.ERROR)
try:
    from transformers.utils import logging as transformers_logging
    transformers_logging.disable_progress_bar()
except ImportError:
    pass

import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

print('ChromaDB:', CHROMA_DIR)

In [ ]:
MODELS = {
    'sealion': {'collection': 'legal_articles_sealion', 'model': 'aisingapore/SEA-LION-E5-Embedding-600M'},
    'granite-r2': {'collection': 'legal_articles_granite_r2', 'model': 'ibm-granite/granite-embedding-311m-multilingual-r2'},
}
MODEL_KEYS = ['sealion', 'granite-r2']
TOP_K = 5
print('Models to compare:', MODEL_KEYS)

In [ ]:
def normalize_query(query: str) -> str:
    query = unicodedata.normalize('NFC', query)
    query = query.replace('\u200b', '').replace('\u200c', '').replace('\u200d', '')
    query = query.replace('\u00a0', ' ')
    return ' '.join(query.split())

def embed_query(query: str, model_key: str) -> np.ndarray:
    model_config = MODELS[model_key]
    if model_key == 'albert':
        import torch
        from transformers import AutoModel, AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_config['model'])
        model = AutoModel.from_pretrained(model_config['model'])
        encoded = tokenizer([query], padding=True, truncation=True, max_length=512, return_tensors='pt')
        with torch.no_grad():
            output = model(**encoded).last_hidden_state
        mask = encoded['attention_mask'].unsqueeze(-1).float()
        vector = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        return (vector / vector.norm(dim=1, keepdim=True))[0].cpu().numpy().astype('float32')
    model = SentenceTransformer(model_config['model'])
    vector = model.encode([query], show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
    return np.asarray(vector[0], dtype='float32')

In [ ]:
def semantic_search(query: str, model_key: str, top_k: int = TOP_K) -> list[dict]:
    query = normalize_query(query)
    if not query:
        raise ValueError('Query cannot be empty')
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    collection = client.get_collection(MODELS[model_key]['collection'])
    query_vector = embed_query(query, model_key)
    result = collection.query(query_embeddings=[query_vector.tolist()], n_results=top_k, include=['documents', 'metadatas', 'distances'])
    matches = []
    for rank, article_id in enumerate(result['ids'][0]):
        metadata = result['metadatas'][0][rank] or {}
        matches.append({
            'rank': rank + 1,
            'id': article_id,
            'distance': result['distances'][0][rank],
            'document_id': metadata.get('document_id'),
            'article_number': metadata.get('article_number'),
            'title': metadata.get('document_title'),
            'text': result['documents'][0][rank],
            'source_url': metadata.get('source_url'),
            'pdf_url': metadata.get('pdf_url'),
        })
    return matches